# AgriMind Phase 4: Expert System (Chemical Mixture Rule-Checker)

This component serves as the **Safety Guard (Rule-Checker)** of the **AgriMind Federated Advisory System**.
Before proposing fertilizer, pesticide, fungicide, or herbicide mixtures, the Advisory System checks them against this Expert System to prevent:
1. **Precipitation Reactions**: Calcium and Sulfate/Phosphate mixing in solutions, rendering nutrients unavailable and clogging sprayers.
2. **Alkaline Hydrolysis**: Mixing alkaline products (e.g., Bordeaux Mixture, Lime Sulfur) with organophosphates, destroying pesticide effectiveness.
3. **Phytotoxicity**: Mixtures that cause foliar damage or chemical burn to leaves (e.g., Lime Sulfur + Neem Oil).
4. **Hazardous Reactions**: Gas releases (e.g., toxic $H_2S$ when Lime Sulfur mixes with acidic fertilizers).
5. **Pollinator/Environmental Hazards**: Synergistic toxicity to beneficial insects like honeybees.

### 1. Import Dependencies
We import our newly created `AgriMindExpertSystem` module along with visualization and data reporting libraries.

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from agrimind_expert_system import AgriMindExpertSystem

### 2. Initialize the Expert System
The expert system loads the rule definitions directly from `chemical_rules.json`.

In [ ]:
# Initialize system with our rules JSON
expert_system = AgriMindExpertSystem("chemical_rules.json")

# List all registered chemicals
chems_df = pd.DataFrame(expert_system.list_chemicals())
print(f"Loaded {len(chems_df)} chemicals from registry.")
chems_df.style.set_properties(**{'text-align': 'left'}).set_table_styles([
    dict(selector='th', props=[('text-align', 'left')])
])

### 3. Interactive Validation Testing
We will simulate several proposed mixtures of chemicals and see how the Expert System responds to safe, risky, and hazardous mixtures.

In [ ]:
test_mixtures = {
    "Safe Fertilizer & Fungicide Mix": ["Urea", "Chlorothalonil"],
    "Precipitation Conflict Mix (Calcium & Magnesium)": ["Calcium Nitrate", "Magnesium Sulfate"],
    "Phytotoxic Chemical Mix (Sulfur & Oil)": ["Lime Sulfur", "Neem Oil"],
    "Alkaline Hydrolysis Mix (Copper & Organophosphate)": ["Bordeaux Mixture", "Chlorpyrifos"],
    "Synergistic Bee Toxicity Hazard": ["Imidacloprid", "Captan"],
    "Unregistered Chemical Mixture": ["Calcium Nitrate", "MagicGrowNutrientX"]
}

for name, mix in test_mixtures.items():
    print("=" * 80)
    print(f"TESTING: {name}")
    print(f"Proposed Mixture: {mix}")
    print("-" * 80)
    
    report = expert_system.validate_mixture(mix)
    print(f"STATUS : {report['status']}")
    print(f"SUMMARY: {report['summary']}")
    
    if report['conflicts']:
        print("CONFLICTS DETAILS:")
        for i, conflict in enumerate(report['conflicts'], 1):
            print(f"  {i}. Pair: {conflict['chemical_1']} + {conflict['chemical_2']}")
            print(f"     Type: {conflict['type']} ({conflict['severity']})")
            print(f"     Reason: {conflict['reason']}")
            print(f"     Alternative: {conflict['alternative']}")
    print("\n")

### 4. Visualizing the Rules Network
Below is a network graph displaying chemicals and their incompatibility constraints. Red edges represent **Blocked** combinations, while yellow/orange edges represent **Warnings**.

In [ ]:
def plot_incompatibility_network(system):
    G = nx.Graph()
    
    # Add all chemical nodes
    for chem in system.chemicals.values():
        G.add_node(chem["name"], category=chem["category"])
        
    # Add incompatibility edges
    edge_colors = []
    edge_labels = {}
    for key, conflict in system.incompatibilities.items():
        chem1_orig = None
        chem2_orig = None
        
        # Find exact case-sensitive name
        for name in system.chemicals:
            if name.lower() == key[0]:
                chem1_orig = system.chemicals[name]["name"]
            elif name.lower() == key[1]:
                chem2_orig = system.chemicals[name]["name"]
                
        if chem1_orig and chem2_orig:
            G.add_edge(chem1_orig, chem2_orig, weight=2)
            if conflict["severity"] == "Blocked":
                edge_colors.append("red")
            else:
                edge_colors.append("orange")
            edge_labels[(chem1_orig, chem2_orig)] = conflict["type"]

    # Position nodes using spring layout
    plt.figure(figsize=(14, 10))
    pos = nx.spring_layout(G, k=0.6, seed=42)
    
    # Categorize nodes for styling
    categories = set(nx.get_node_attributes(G, 'category').values())
    color_map = {
        "Fertilizer": "#a2d2ff",
        "Fungicide": "#ffafcc",
        "Insecticide": "#ccff33",
        "Herbicide": "#ffc6ff"
    }
    
    for cat in categories:
        nodes = [n for n, attr in G.nodes(data=True) if attr.get('category') == cat]
        nx.draw_networkx_nodes(G, pos, nodelist=nodes, node_color=color_map.get(cat, 'gray'),
                               node_size=2000, label=cat, edgecolors='black')

    # Draw edges with severity-based coloring
    edges = G.edges()
    nx.draw_networkx_edges(G, pos, width=2.5, edge_color=edge_colors)
    
    # Labels
    nx.draw_networkx_labels(G, pos, font_size=9, font_family="sans-serif", font_weight="bold")
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8, font_color="#555555")
    
    plt.title("AgriMind Chemical Compatibility & Incompatibility Network", fontsize=16, fontweight="bold", pad=20)
    plt.legend(scatterpoints=1, loc='upper left', frameon=True, facecolor='white', edgecolor='black')
    plt.axis("off")
    plt.tight_layout()
    plt.show()

plot_incompatibility_network(expert_system)

### 5. Automated Validation Tests
We implement unit tests to run and verify the validation engine works correctly.

In [ ]:
import unittest

class TestAgriMindExpertSystem(unittest.TestCase):
    def setUp(self):
        self.system = AgriMindExpertSystem("chemical_rules.json")
        
    def test_safe_mixture(self):
        # Urea + Chlorothalonil is safe
        report = self.system.validate_mixture(["Urea", "Chlorothalonil"])
        self.assertEqual(report["status"], "Approved")
        self.assertEqual(len(report["conflicts"]), 0)
        
    def test_blocked_precipitation_mixture(self):
        # Calcium Nitrate + Magnesium Sulfate is blocked
        report = self.system.validate_mixture(["Calcium Nitrate", "Magnesium Sulfate"])
        self.assertEqual(report["status"], "Blocked")
        self.assertTrue(any(c["type"] == "Precipitation" for c in report["conflicts"]))
        
    def test_alkaline_hydrolysis(self):
        # Bordeaux Mixture + Chlorpyrifos is blocked
        report = self.system.validate_mixture(["Bordeaux Mixture", "Chlorpyrifos"])
        self.assertEqual(report["status"], "Blocked")
        self.assertTrue(any(c["type"] == "Alkaline Hydrolysis" for c in report["conflicts"]))
        
    def test_warning_bee_toxicity(self):
        # Imidacloprid + Captan is warning
        report = self.system.validate_mixture(["Imidacloprid", "Captan"])
        self.assertEqual(report["status"], "Warning")
        self.assertTrue(any(c["type"] == "Synergistic Bee Toxicity" for c in report["conflicts"]))
        
    def test_unregistered_chemical(self):
        # Unknown chemical throws warning
        report = self.system.validate_mixture(["Calcium Nitrate", "InvalidChem123"])
        self.assertEqual(report["status"], "Warning")
        self.assertTrue(any(c["type"] == "Unregistered Chemical Check" for c in report["conflicts"]))

# Run the tests in the notebook
suite = unittest.TestLoader().loadTestsFromTestCase(TestAgriMindExpertSystem)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)